# CYMEK FORMATION-MUX-001 — Kaggle T4 x2 campaign

## BEFORE RUNNING

```
Kaggle Settings:
  Accelerator -> GPU T4 x2
  Internet    -> ON
```

- Do NOT choose TPU for this campaign. Do NOT choose CPU.
- Official execution requires exactly two visible T4-class GPUs; the notebook
  stops before any science otherwise (a manual debug mode is labeled
  ENGINEERING_ONLY and cannot produce an official verdict).
- Prefer Save Version -> Save & Run All for the full campaign (12h ceiling).
- One operational run path: hardware gate -> qualification -> calibration ->
  campaign (24 official arms on two independent GPU workers) -> sealed
  finalization -> FORMATION_MUX_001_RESULTS.zip in the notebook Output.


In [ ]:
# CELL 1 — environment, hardware gate, frozen science checkout
import json, subprocess, sys
from pathlib import Path
REPO = Path('/kaggle/working/An-Ra-the-new-AGI')
REMOTE = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
BRANCH = 'cymek-next-core-architecture'
SCIENCE_COMMIT = '36ab16a4950d582fcc26b239dfc8c0ba816911bb'
SCIENCE_BLOBS = {"anra_v5/formation_mux_model.py": "1928937192ad1c0709f45a41a9fa33365714d8b65552366d5167c84e575d8e79", "anra_v5/formation_mux_train.py": "281f384ea9049974d29b47af7b2e65865b3403d8cd852c0e3912dd7aca787b21", "v5_experiments/formation_mux_protocol.py": "981fa001e90aca43c2c46a076317f7557713341ddd6500214af5f160d32358ec", "v5_experiments/formation_mux_data.py": "758367ce2ab3dec9307d1c0a8b8c56424c8353cdb3927f974a369a1941ae40f5", "tools/formation_mux_001_worker.py": "b581622a8bee812545effa78b10678032bdddc89877e724e02b5baffdee05ec1", "docs/cymek/experiments/CS-MECH-002/PREREGISTRATION.json": "2b6dac20adb2899630b875681221b006477f228d04956682dc3282831ea89cae", "docs/cymek/experiments/REP-FORM-003A/PREREGISTRATION.json": "5e04d29f624827d43545a2385cc2a62e92b6092345d9515abf07a6fd5d5c1777"}
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REMOTE, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', SCIENCE_COMMIT], check=True)
head = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
assert head == SCIENCE_COMMIT, f'HEAD {head} != science commit'
import hashlib
for rel, expected in SCIENCE_BLOBS.items():
    actual = hashlib.sha256((REPO / rel).read_bytes()).hexdigest()
    assert actual == expected, f'science blob mismatch: {rel}'
print('science commit + ' + str(len(SCIENCE_BLOBS)) + ' blob hashes verified')
import subprocess as sp
need = []
try:
    import tokenizers  # noqa
except ImportError:
    need.append('tokenizers')
if need:
    sp.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True)
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda,
      '| devices', torch.cuda.device_count())


In [ ]:
# CELL 2 — one canonical operator run (gate -> qualification -> calibration -> campaign -> sealed -> package)
import sys
from pathlib import Path
REPO = Path('/kaggle/working/An-Ra-the-new-AGI')
sys.path.insert(0, str(REPO))
import subprocess
code = subprocess.run([sys.executable, 'tools/formation_mux_001_kaggle_operator.py',
                       '--repo', str(REPO),
                       '--out', '/kaggle/working/FORMATION_MUX_001'],
                      cwd=str(REPO))
if code != 0:
    raise RuntimeError(f'operator exited {code}: see receipts under /kaggle/working/FORMATION_MUX_001')


In [ ]:
# CELL 3 — retrieve results
import json, hashlib
from pathlib import Path
bundle = Path('/kaggle/working/FORMATION_MUX_001_RESULTS.zip')
print('BUNDLE:', bundle, hashlib.sha256(bundle.read_bytes()).hexdigest()[:16])
state = json.loads(Path('/kaggle/working/FORMATION_MUX_001/CAMPAIGN_STATE.json').read_text())
print('STATUS:', state.get('status'), '| arms complete:',
      state.get('summary', {}).get('complete_arms'), '/', state.get('summary', {}).get('required_arms'))
for experiment in ('CS-MECH-002', 'REP-FORM-003A'):
    final = Path('/kaggle/working/FORMATION_MUX_001') / experiment / 'FINAL_RESULT.json'
    print(experiment, '->', json.loads(final.read_text())['verdict'] if final.exists() else 'sealed not consumed (see campaign state)')
